In [17]:
import pandas as pd
import numpy as np

# -------------------------------
# 1 Load Dataset
# -------------------------------

df = pd.read_csv(r"C:\Users\DELL\Desktop\ipl_matchup\data\raw\ball_by_ball_ipl.csv")

print("Initial Shape:", df.shape)

# -------------------------------
# 2 Remove Unwanted Columns
# -------------------------------

df = df.loc[:, ~df.columns.str.contains("Unnamed")]

print("After removing unnamed columns:", df.shape)

# -------------------------------
# 3 Standardize Column Names
# -------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nColumn Names After Cleaning:\n")
print(df.columns)



# -------------------------------
# 5 Verify Required Columns
# -------------------------------

required_columns = [
    "batter",
    "bowler",
    "batting_team",
    "bowling_team",
    "over",
    "ball",
    "total_batsman_runs"
]

available_columns = [col for col in required_columns if col in df.columns]

print("\nAvailable Important Columns:\n", available_columns)

# -------------------------------
# 6 Remove Rows With Missing Key Values
# -------------------------------

df = df.dropna(subset=["batter", "bowler"])

print("\nShape After Cleaning:", df.shape)

# -------------------------------
# 7 Quick Data Insights
# -------------------------------

print("\nTop Batsmen By Runs")

print("\nTop Bowlers By Wickets (if column exists)")
if "player_dismissed" in df.columns:
    print(df.groupby("bowler")["player_dismissed"].count().sort_values(ascending=False).head(10))

# -------------------------------
# 8 Save Clean Dataset
# -------------------------------

df.to_csv("../data/processed/clean_ipl_data.csv", index=False)

print("\nClean dataset saved to data/processed/clean_ipl_data.csv")

print("\nFinal Columns Used:")
print(df.columns)

Initial Shape: (239693, 35)
After removing unnamed columns: (239693, 34)

Column Names After Cleaning:

Index(['match_id', 'date', 'venue', 'bat_first', 'bat_second', 'innings',
       'over', 'ball', 'batter', 'non_striker', 'bowler', 'batter_runs',
       'extra_runs', 'runs_from_ball', 'ball_rebowled', 'extra_type', 'wicket',
       'method', 'player_out', 'innings_runs', 'innings_wickets',
       'target_score', 'runs_to_get', 'balls_remaining', 'winner',
       'chased_successfully', 'total_batter_runs', 'total_non_striker_runs',
       'batter_balls_faced', 'non_striker_balls_faced', 'player_out_runs',
       'player_out_balls_faced', 'bowler_runs_conceded', 'valid_ball'],
      dtype='object')

Available Important Columns:
 ['batter', 'bowler', 'over', 'ball']

Shape After Cleaning: (239693, 34)

Top Batsmen By Runs

Top Bowlers By Wickets (if column exists)

Clean dataset saved to data/processed/clean_ipl_data.csv

Final Columns Used:
Index(['match_id', 'date', 'venue', 'bat_fi

In [21]:
# This modifies the dataframe directly in memory
df.rename(columns={'batter': 'batsman','bat_first':'batting_team','bat_second':'bowling_team','batter_runs':'batsman_runs'}, inplace=True)

In [12]:
df.rename(columns={'player_out':'player_dismissed'}, inplace=True)

In [13]:
df.columns

Index(['match_id', 'date', 'venue', 'batting_team', 'bowling_team', 'innings',
       'over', 'ball', 'batsman', 'non_striker', 'bowler', 'batsman_runs',
       'extra_runs', 'runs_from_ball', 'ball_rebowled', 'extra_type', 'wicket',
       'method', 'player_dismissed', 'innings_runs', 'innings_wickets',
       'target_score', 'runs_to_get', 'balls_remaining', 'winner',
       'chased_successfully', 'total_batter_runs', 'total_non_striker_runs',
       'batter_balls_faced', 'non_striker_balls_faced', 'player_out_runs',
       'player_out_balls_faced', 'bowler_runs_conceded', 'valid_ball',
       'runs_vs_bowler', 'balls_vs_bowler', 'dismissals',
       'strike_rate_vs_bowler', 'dismissal_rate', 'bowler_economy_x',
       'bowler_economy_y', 'match_phase'],
      dtype='object')

In [4]:
import pandas as pd

df = pd.read_csv(r"C:\Users\DELL\Desktop\ipl_matchup\data\processed\clean_ipl_data.csv")

# batsman vs bowler history
matchup_stats = (
    df.groupby(["batsman","bowler"])
    .agg(
        runs_vs_bowler=("batsman_runs","sum"),
        balls_vs_bowler=("batsman_runs","count"),
        dismissals=("player_out","count")
    )
    .reset_index()
)

matchup_stats["strike_rate_vs_bowler"] = (
    matchup_stats["runs_vs_bowler"] /
    matchup_stats["balls_vs_bowler"]
) * 100

matchup_stats["dismissal_rate"] = (
    matchup_stats["dismissals"] /
    matchup_stats["balls_vs_bowler"]
)

matchup_stats.head()

,batsman,bowler,runs_vs_bowler,balls_vs_bowler,dismissals,strike_rate_vs_bowler,dismissal_rate
0,A Ashish Reddy,A Nehra,7,9,1,77.777778,0.111111
1,A Ashish Reddy,AB Dinda,9,7,0,128.571429,0.000000
2,A Ashish Reddy,AD Mathews,25,12,0,208.333333,0.000000
3,A Ashish Reddy,AD Russell,4,3,1,133.333333,0.333333
4,A Ashish Reddy,Anureet Singh,2,2,0,100.000000,0.000000


In [5]:
df = df.merge(matchup_stats,
              on=["batsman","bowler"],
              how="left")

In [6]:
bowler_stats = (
    df.groupby("bowler")
    .agg(
        bowler_runs=("batsman_runs","sum"),
        balls_bowled=("batsman_runs","count")
    )
    .reset_index()
)

bowler_stats["bowler_economy"] = (
    bowler_stats["bowler_runs"] /
    bowler_stats["balls_bowled"]
) * 6

In [7]:
df = df.merge(bowler_stats[["bowler","bowler_economy"]],
              on="bowler",
              how="left")

In [ ]:
df["outcome"] = "dot"

df.loc[df["batsman_runs"] == 1, "outcome"] = "single"
df.loc[df["batsman_runs"] == 2, "outcome"] = "double"
df.loc[df["batsman_runs"] == 3, "outcome"] = "triple"
df.loc[df["batsman_runs"] == 4, "outcome"] = "four"
df.loc[df["batsman_runs"] == 6, "outcome"] = "six"

df.loc[df["player_dismissed"].notna(), "outcome"] = "wicket"




In [19]:
batsman_stats = (
    df.groupby("batsman")
    .agg(
        batsman_runs=("batsman_runs","sum"),
        balls_faced=("batsman_runs","count")
    )
    .reset_index()
)

batsman_stats["batsman_strike_rate"] = (
    batsman_stats["batsman_runs"] /
    batsman_stats["balls_faced"]
) * 100

In [20]:
df = df.merge(
    batsman_stats[["batsman","batsman_strike_rate"]],
    on="batsman",
    how="left"
)

In [21]:
venue_stats = (
    df.groupby("venue")
    .agg(
        avg_runs=("batsman_runs","mean")
    )
    .reset_index()
)

df = df.merge(venue_stats, on="venue", how="left")

In [16]:
def get_phase(over):
    if over <= 6:
        return "powerplay"
    elif over <= 15:
        return "middle"
    else:
        return "death"

df["match_phase"] = df["over"].apply(get_phase)

In [24]:
bowler_stats = (
    df.groupby("bowler")
    .agg(
        bowler_runs=("batsman_runs","sum"),
        balls_bowled=("batsman_runs","count")
    )
    .reset_index()
)

bowler_stats["bowler_economy"] = (
    bowler_stats["bowler_runs"] /
    bowler_stats["balls_bowled"]
) * 6

bowler_stats.head()

,bowler,bowler_runs,balls_bowled,bowler_economy
0,A Ashish Reddy,379,264,8.613636
1,A Badoni,11,13,5.076923
2,A Chandila,242,234,6.205128
3,A Choudhary,137,108,7.611111
4,A Dananjaya,46,25,11.040000


In [27]:
df.drop(columns=['bowler_economy_x','bowler_economy_y',])

,match_id,date,venue,batting_team,bowling_team,innings,over,ball,batsman,non_striker,...,valid_ball,runs_vs_bowler,balls_vs_bowler,dismissals,strike_rate_vs_bowler,dismissal_rate,match_phase,outcome,batsman_strike_rate,avg_runs
0,1359507,2023-04-23,Eden Gardens,Chennai Super Kings,Kolkata Knight Riders,1,1,1,RD Gaikwad,DP Conway,...,1,12,13,1,92.307692,0.076923,powerplay,four,132.522124,1.268424
1,1359507,2023-04-23,Eden Gardens,Chennai Super Kings,Kolkata Knight Riders,1,1,2,RD Gaikwad,DP Conway,...,1,12,13,1,92.307692,0.076923,powerplay,dot,132.522124,1.268424
2,1359507,2023-04-23,Eden Gardens,Chennai Super Kings,Kolkata Knight Riders,1,1,3,RD Gaikwad,DP Conway,...,1,12,13,1,92.307692,0.076923,powerplay,dot,132.522124,1.268424
3,1359507,2023-04-23,Eden Gardens,Chennai Super Kings,Kolkata Knight Riders,1,1,4,RD Gaikwad,DP Conway,...,1,12,13,1,92.307692,0.076923,powerplay,dot,132.522124,1.268424
4,1359507,2023-04-23,Eden Gardens,Chennai Super Kings,Kolkata Knight Riders,1,1,5,RD Gaikwad,DP Conway,...,1,12,13,1,92.307692,0.076923,powerplay,single,132.522124,1.268424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239688,1136596,2018-05-05,Rajiv Gandhi International Stadium,Delhi Daredevils,Sunrisers Hyderabad,2,20,1,YK Pathan,KS Williamson,...,1,17,7,0,242.857143,0.000000,death,double,137.001733,1.236646
239689,1136596,2018-05-05,Rajiv Gandhi International Stadium,Delhi Daredevils,Sunrisers Hyderabad,2,20,2,YK Pathan,KS Williamson,...,1,17,7,0,242.857143,0.000000,death,six,137.001733,1.236646
239690,1136596,2018-05-05,Rajiv Gandhi International Stadium,Delhi Daredevils,Sunrisers Hyderabad,2,20,3,YK Pathan,KS Williamson,...,1,17,7,0,242.857143,0.000000,death,four,137.001733,1.236646
239691,1136596,2018-05-05,Rajiv Gandhi International Stadium,Delhi Daredevils,Sunrisers Hyderabad,2,20,4,YK Pathan,KS Williamson,...,1,17,7,0,242.857143,0.000000,death,single,137.001733,1.236646


In [28]:
df.to_csv(r"C:\Users\DELL\Desktop\ipl_matchup\data\processed\clean_ipl_data.csv", index=False)

In [29]:
df.columns

Index(['match_id', 'date', 'venue', 'batting_team', 'bowling_team', 'innings',
       'over', 'ball', 'batsman', 'non_striker', 'bowler', 'batsman_runs',
       'extra_runs', 'runs_from_ball', 'ball_rebowled', 'extra_type', 'wicket',
       'method', 'player_dismissed', 'innings_runs', 'innings_wickets',
       'target_score', 'runs_to_get', 'balls_remaining', 'winner',
       'chased_successfully', 'total_batter_runs', 'total_non_striker_runs',
       'batter_balls_faced', 'non_striker_balls_faced', 'player_out_runs',
       'player_out_balls_faced', 'bowler_runs_conceded', 'valid_ball',
       'runs_vs_bowler', 'balls_vs_bowler', 'dismissals',
       'strike_rate_vs_bowler', 'dismissal_rate', 'bowler_economy_x',
       'bowler_economy_y', 'match_phase', 'outcome', 'batsman_strike_rate',
       'avg_runs'],
      dtype='object')